# Glance Fashion Retrieval - Notebook 2: Retriever & Evaluation
Run **after** Notebook 1 has completed and the `vector_db` directory is populated.
Attach the same two datasets as Notebook 1.

In [ ]:
import os, shutil, sys, warnings
warnings.filterwarnings('ignore')

# Suppress HuggingFace verbose model-load messages
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

code_source = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'config.py' in os.listdir(os.path.join(root, 'src')):
        code_source = root
        break

work_path = '/kaggle/working/glance-code'
if not os.path.exists(work_path) and code_source:
    shutil.copytree(code_source, work_path)
    print('Copied code to: ' + work_path)
else:
    print('Using code at: ' + work_path)

if work_path not in sys.path:
    sys.path.insert(0, work_path)

print('Setup complete!')

In [ ]:
!pip install -q -r /kaggle/working/glance-code/requirements.txt
!pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
from src.retriever import search, search_without_reranker, _print_results
from src.config import EVAL_QUERIES
from src.utils import display_results, display_ablation_comparison, create_ablation_table
from IPython.display import display, Markdown

print('Imports OK. Running evaluation on', len(EVAL_QUERIES), 'queries...')

## 1. Full Pipeline Evaluation

In [ ]:
full_results = {}
for q in EVAL_QUERIES:
    print('\n--- Query:', q, '---')
    results = search(q, top_n=5)
    full_results[q] = results
    _print_results(q, results)
    fig = display_results(q, results)
    display(fig)

## 2. Ablation Study: Bi-Encoder Only vs Full Pipeline

In [ ]:
baseline_results = {}
for q in EVAL_QUERIES:
    baseline_results[q] = search_without_reranker(q, top_n=5)

ablation_data = {
    'FashionSigLIP Only': baseline_results,
    'Full Pipeline (SigLIP + BLIP)': full_results,
}

table = create_ablation_table(EVAL_QUERIES, ablation_data)
display(Markdown(table))

## 3. Visual Side-by-Side Comparison (Hardest Query)

In [ ]:
# Visual comparison on the most compositional query
hard_query = EVAL_QUERIES[-1]
print('Visual comparison for:', hard_query)
fig = display_ablation_comparison(
    hard_query,
    baseline_results[hard_query],
    full_results[hard_query]
)
display(fig)